In [1]:

# Cell 1: Setup and imports
from __future__ import annotations

import csv
import math
import os

# Local macOS conda environments can load duplicate OpenMP runtimes via Torch/SCIP.
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")

import pickle
import random
import sys
import time
from pathlib import Path

import ecole
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy
import torch
import torch_geometric
from scipy.stats import pearsonr, spearmanr

ROOT = Path.cwd().resolve()
DATA_DIR = ROOT / "data" / "instances"
RESULTS_DIR = ROOT / "results"
MODEL_DIR = ROOT / "models" / "baseline_setcover"
MODEL_PATH = MODEL_DIR / "train_params.pkl"
RESULTS_DIR.mkdir(exist_ok=True)

EVAL_CLASS_DIRS = {
    "setcover_valid": DATA_DIR / "eval_setcover",
    "cauctions": DATA_DIR / "eval_cauctions",
    "facilities": DATA_DIR / "eval_facilities",
    "indset": DATA_DIR / "eval_indset",
    "mknapsack": DATA_DIR / "eval_mknapsack",
    "setpacking": DATA_DIR / "eval_setpacking",
    "setpartitioning": DATA_DIR / "eval_setpartitioning",
    "vertexcover": DATA_DIR / "eval_vertexcover",
    "binpacking": DATA_DIR / "eval_binpacking",
    "generalized_assignment": DATA_DIR / "eval_generalized_assignment",
}
REFERENCE_DIR = DATA_DIR / "reference_setcover"

for path in [ROOT]:
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

print(f"root: {ROOT}")
print(f"data instances: {DATA_DIR} exists={DATA_DIR.exists()}")
print(f"model: {MODEL_PATH} exists={MODEL_PATH.exists()}")
print(f"ecole: {ecole.__version__}")
print(f"torch: {torch.__version__}")
print(f"torch_geometric: {torch_geometric.__version__}")
print(f"scipy: {scipy.__version__}")
print(f"cuda available: {torch.cuda.is_available()}")


root: /Users/skandavyassrinivasan/Documents/Integral-Optimization-Branch-Learning-Research
data instances: /Users/skandavyassrinivasan/Documents/Integral-Optimization-Branch-Learning-Research/data/instances exists=True
model: /Users/skandavyassrinivasan/Documents/Integral-Optimization-Branch-Learning-Research/models/baseline_setcover/train_params.pkl exists=True
ecole: 0.8.1
torch: 2.12.0
torch_geometric: 2.7.0
scipy: 1.17.1
cuda available: False


In [2]:

# Cell 2: Instance inventory
# The expensive experiment uses curated MILP instances committed under data/instances/.
# If these folders are missing, run:
#   conda run -n ecole python -m scripts.prepare_experiment_instances --count 50

TARGET_PER_CLASS = 50
REFERENCE_SIZE = 50


def instance_files(path: Path) -> list[Path]:
    return sorted(list(path.glob("*.lp")) + list(path.glob("*.mps")))

inventory_rows = []
reference_count = len(instance_files(REFERENCE_DIR))
inventory_rows.append({"role": "reference", "class": "setcover", "folder": str(REFERENCE_DIR), "n_instances": reference_count})
for class_name, folder in EVAL_CLASS_DIRS.items():
    inventory_rows.append({"role": "eval", "class": class_name, "folder": str(folder), "n_instances": len(instance_files(folder))})

inventory_df = pd.DataFrame(inventory_rows)
inventory_df.to_csv(RESULTS_DIR / "instance_inventory.csv", index=False)
print(inventory_df.to_string(index=False))

missing = inventory_df[
    ((inventory_df["role"] == "reference") & (inventory_df["n_instances"] < REFERENCE_SIZE))
    | ((inventory_df["role"] == "eval") & (inventory_df["n_instances"] < TARGET_PER_CLASS))
]
if not missing.empty:
    raise RuntimeError(
        "Missing curated instances. Run: conda run -n ecole python -m scripts.prepare_experiment_instances --count 50\n"
        + missing.to_string(index=False)
    )


     role                  class                                                                                                                          folder  n_instances
reference               setcover          /Users/skandavyassrinivasan/Documents/Integral-Optimization-Branch-Learning-Research/data/instances/reference_setcover           50
     eval         setcover_valid               /Users/skandavyassrinivasan/Documents/Integral-Optimization-Branch-Learning-Research/data/instances/eval_setcover           50
     eval              cauctions              /Users/skandavyassrinivasan/Documents/Integral-Optimization-Branch-Learning-Research/data/instances/eval_cauctions           50
     eval             facilities             /Users/skandavyassrinivasan/Documents/Integral-Optimization-Branch-Learning-Research/data/instances/eval_facilities           50
     eval                 indset                 /Users/skandavyassrinivasan/Documents/Integral-Optimization-Branch-Learning-Resea

In [3]:

# Cell 3: Load model
if not MODEL_PATH.exists():
    raise FileNotFoundError(f"Missing trained model weights: {MODEL_PATH}")

from models.learn2branch_gnn import GNNPolicy

device = "cuda:0" if torch.cuda.is_available() else "cpu"
policy = GNNPolicy().to(device)
policy.load_state_dict(torch.load(MODEL_PATH, map_location=device))
policy.eval()

print(f"loaded GNNPolicy from models.learn2branch_gnn")
print(f"loaded weights from {MODEL_PATH}")
print(f"device: {device}")


loaded GNNPolicy from models.learn2branch_gnn
loaded weights from /Users/skandavyassrinivasan/Documents/Integral-Optimization-Branch-Learning-Research/models/baseline_setcover/train_params.pkl
device: cpu


In [4]:

# Cell 4: Load distance metric and build reference set
from milp_distance.distance import extract_normalized_representation, greedy_instance_distance

REFERENCE_SEED = 0
REFERENCE_CACHE = RESULTS_DIR / "reference_reps.pkl"
DISTANCE_CACHE_CSV = RESULTS_DIR / "distance_cache.csv"

reference_candidates = instance_files(REFERENCE_DIR)
if len(reference_candidates) < REFERENCE_SIZE:
    raise RuntimeError(f"Need at least {REFERENCE_SIZE} reference setcover instances in {REFERENCE_DIR}; found {len(reference_candidates)}")

rng_ref = np.random.RandomState(REFERENCE_SEED)
reference_paths = [reference_candidates[i] for i in rng_ref.choice(len(reference_candidates), size=REFERENCE_SIZE, replace=False)]
print(f"using {len(reference_paths)} reference instances from {REFERENCE_DIR}")

if REFERENCE_CACHE.exists():
    with REFERENCE_CACHE.open("rb") as handle:
        payload = pickle.load(handle)
    if payload.get("paths") == [str(p.resolve()) for p in reference_paths]:
        reference_reps = payload["reps"]
        print(f"loaded cached reference representations from {REFERENCE_CACHE}")
    else:
        reference_reps = None
else:
    reference_reps = None

if reference_reps is None:
    reference_reps = []
    for i, path in enumerate(reference_paths, start=1):
        print(f"reference {i}/{REFERENCE_SIZE}: {path.name}")
        reference_reps.append(extract_normalized_representation(path))
    with REFERENCE_CACHE.open("wb") as handle:
        pickle.dump({"paths": [str(p.resolve()) for p in reference_paths], "reps": reference_reps}, handle)

if DISTANCE_CACHE_CSV.exists():
    distance_cache_df = pd.read_csv(DISTANCE_CACHE_CSV)
    distance_cache = dict(zip(distance_cache_df["path"], distance_cache_df["distance"]))
else:
    distance_cache = {}


def save_distance_cache():
    pd.DataFrame(
        [{"path": path, "distance": distance} for path, distance in sorted(distance_cache.items())]
    ).to_csv(DISTANCE_CACHE_CSV, index=False)


def distribution_distance(mps_path) -> float:
    path = str(Path(mps_path).resolve())
    if path in distance_cache:
        return float(distance_cache[path])
    rep = extract_normalized_representation(path)
    distances = [greedy_instance_distance(rep, ref) for ref in reference_reps if ref is not None]
    finite = [d for d in distances if np.isfinite(d)]
    distance = float(np.mean(finite)) if finite else float("inf")
    distance_cache[path] = distance
    if len(distance_cache) % 10 == 0:
        save_distance_cache()
    return distance

print(f"reference reps ready: {len(reference_reps)}")
print(f"distance cache entries: {len(distance_cache)}")


using 50 reference instances from /Users/skandavyassrinivasan/Documents/Integral-Optimization-Branch-Learning-Research/data/instances/reference_setcover
reference 1/50: setcover_reference_028.lp
reference 2/50: setcover_reference_011.lp
reference 3/50: setcover_reference_010.lp
reference 4/50: setcover_reference_041.lp
reference 5/50: setcover_reference_002.lp
reference 6/50: setcover_reference_027.lp
reference 7/50: setcover_reference_038.lp
reference 8/50: setcover_reference_031.lp
reference 9/50: setcover_reference_022.lp
reference 10/50: setcover_reference_004.lp
reference 11/50: setcover_reference_033.lp
reference 12/50: setcover_reference_035.lp
reference 13/50: setcover_reference_026.lp
reference 14/50: setcover_reference_034.lp
reference 15/50: setcover_reference_018.lp
reference 16/50: setcover_reference_007.lp
reference 17/50: setcover_reference_014.lp
reference 18/50: setcover_reference_045.lp
reference 19/50: setcover_reference_048.lp
reference 20/50: setcover_reference_029

In [5]:

# Cell 5: Branching evaluation function
SCIP_PARAMS = {
    "separating/maxrounds": 0,
    "presolving/maxrestarts": 0,
    "limits/time": 60.0,
    "limits/nodes": 10000,
    "timing/clocktype": 1,
}


def tensorize_node_observation(obs, device):
    return (
        torch.from_numpy(obs.row_features.astype(np.float32)).to(device),
        torch.from_numpy(obs.edge_features.indices.astype(np.int64)).to(device),
        torch.from_numpy(obs.edge_features.values.astype(np.float32)).view(-1, 1).to(device),
        torch.from_numpy(obs.variable_features.astype(np.float32)).to(device),
    )


def solve_vanilla_scip(instance_path, time_limit=60):
    params = dict(SCIP_PARAMS)
    params["limits/time"] = float(time_limit)
    env = ecole.environment.Configuring(scip_params=params)
    wall_start = time.perf_counter()
    env.reset(str(instance_path))
    _, _, _, _, _ = env.step({})
    walltime = time.perf_counter() - wall_start
    model = env.model.as_pyscipopt()
    return {
        "scip_nodes": int(model.getNNodes()),
        "scip_lps": int(model.getNLPs()),
        "scip_time": float(model.getSolvingTime()),
        "scip_walltime": float(walltime),
        "scip_status": str(model.getStatus()),
        "scip_gap": float(model.getGap()),
    }


def solve_ml_branching(instance_path, policy, device, time_limit=60):
    params = dict(SCIP_PARAMS)
    params["limits/time"] = float(time_limit)
    env = ecole.environment.Branching(
        observation_function=ecole.observation.NodeBipartite(),
        scip_params=params,
        pseudo_candidates=False,
    )
    env.seed(0)
    torch.manual_seed(0)
    wall_start = time.perf_counter()

    obs, action_set, _, done, _ = env.reset(str(instance_path))
    while not done:
        if action_set is None or len(action_set) == 0:
            break
        with torch.no_grad():
            tensors = tensorize_node_observation(obs, device)
            logits = policy(*tensors)
            action_tensor_index = torch.as_tensor(action_set.astype(np.int64), device=device)
            selected = int(torch.argmax(logits[action_tensor_index]).item())
            action = int(action_set[selected])
        obs, action_set, _, done, _ = env.step(action)

    walltime = time.perf_counter() - wall_start
    model = env.model.as_pyscipopt()
    return {
        "ml_nodes": int(model.getNNodes()),
        "ml_lps": int(model.getNLPs()),
        "ml_time": float(model.getSolvingTime()),
        "ml_walltime": float(walltime),
        "ml_status": str(model.getStatus()),
        "ml_gap": float(model.getGap()),
    }


def evaluate_instance(mps_path, policy, device, time_limit=60) -> dict | None:
    try:
        path = Path(mps_path)
        scip = solve_vanilla_scip(path, time_limit=time_limit)
        ml = solve_ml_branching(path, policy, device, time_limit=time_limit)
        scip_nodes = max(1, int(scip["scip_nodes"]))
        ml_nodes = int(ml["ml_nodes"])
        return {
            **scip,
            **ml,
            "degradation": float(ml_nodes / scip_nodes),
        }
    except Exception as exc:
        print(f"FAILED {mps_path}: {type(exc).__name__}: {exc}")
        return None


In [ ]:

# Cell 6: Run experiment
RAW_RESULTS_CSV = RESULTS_DIR / "raw_results.csv"
FAILURES_CSV = RESULTS_DIR / "failures.csv"
N_PER_CLASS = 50
TIME_LIMIT = 60


def first_n(paths, n=N_PER_CLASS):
    return [Path(p) for p in sorted(paths)[:n]]


test_sets = {class_name: first_n(instance_files(folder), N_PER_CLASS) for class_name, folder in EVAL_CLASS_DIRS.items()}
test_sets = {name: paths for name, paths in test_sets.items() if paths}
for name, paths in test_sets.items():
    print(f"{name}: {len(paths)} instances")

if RAW_RESULTS_CSV.exists():
    results_df = pd.read_csv(RAW_RESULTS_CSV)
    completed = set(results_df["path"].astype(str))
    rows = results_df.to_dict("records")
    print(f"loaded {len(rows)} existing results from {RAW_RESULTS_CSV}")
else:
    completed = set()
    rows = []

if FAILURES_CSV.exists():
    failures_df = pd.read_csv(FAILURES_CSV)
    failed_paths = set(failures_df["path"].astype(str))
    failure_rows = failures_df.to_dict("records")
else:
    failed_paths = set()
    failure_rows = []

for class_name, paths in test_sets.items():
    for idx, path in enumerate(paths, start=1):
        resolved = str(path.resolve())
        if resolved in completed or resolved in failed_paths:
            continue
        try:
            distance = distribution_distance(path)
            metrics = evaluate_instance(path, policy, device, time_limit=TIME_LIMIT)
            if metrics is None:
                failure_rows.append({"class": class_name, "path": resolved, "reason": "evaluate_instance returned None"})
                pd.DataFrame(failure_rows).to_csv(FAILURES_CSV, index=False)
                failed_paths.add(resolved)
                continue
            row = {
                "class": class_name,
                "path": resolved,
                "distance": distance,
                "scip_nodes": metrics["scip_nodes"],
                "ml_nodes": metrics["ml_nodes"],
                "degradation": metrics["degradation"],
                "scip_lps": metrics["scip_lps"],
                "ml_lps": metrics["ml_lps"],
                "scip_time": metrics["scip_time"],
                "ml_time": metrics["ml_time"],
                "scip_walltime": metrics["scip_walltime"],
                "ml_walltime": metrics["ml_walltime"],
                "scip_status": metrics["scip_status"],
                "ml_status": metrics["ml_status"],
                "scip_gap": metrics["scip_gap"],
                "ml_gap": metrics["ml_gap"],
            }
            rows.append(row)
            completed.add(resolved)
            pd.DataFrame(rows).to_csv(RAW_RESULTS_CSV, index=False)
            save_distance_cache()
            if idx % 10 == 0 or idx == 1:
                print(f"{class_name} {idx}/{len(paths)} distance={distance:.4f} degradation={metrics['degradation']:.3f}")
        except Exception as exc:
            print(f"FAILED {class_name} {path.name}: {type(exc).__name__}: {exc}")
            failure_rows.append({"class": class_name, "path": resolved, "reason": f"{type(exc).__name__}: {exc}"})
            pd.DataFrame(failure_rows).to_csv(FAILURES_CSV, index=False)
            failed_paths.add(resolved)

results_df = pd.DataFrame(rows)
results_df.to_csv(RAW_RESULTS_CSV, index=False)
save_distance_cache()
print(f"saved {len(results_df)} rows to {RAW_RESULTS_CSV}")
if failure_rows:
    print(f"saved {len(failure_rows)} failures to {FAILURES_CSV}")


setcover_valid: 50 instances
cauctions: 50 instances
facilities: 50 instances
indset: 50 instances
mknapsack: 50 instances
setpacking: 50 instances
setpartitioning: 50 instances
vertexcover: 50 instances
binpacking: 50 instances
generalized_assignment: 50 instances
setcover_valid 1/50 distance=0.0064 degradation=1.912


In [ ]:

# Cell 7: Analysis and plots
RAW_RESULTS_CSV = RESULTS_DIR / "raw_results.csv"
FAILURES_CSV = RESULTS_DIR / "failures.csv"
if not RAW_RESULTS_CSV.exists():
    raise RuntimeError("No raw results found. Run Cell 6 first.")

results_df = pd.read_csv(RAW_RESULTS_CSV)
results_df = results_df.replace([np.inf, -np.inf], np.nan).dropna(subset=["distance", "degradation"])
results_df = results_df[results_df["degradation"] > 0].copy()
if results_df.empty:
    raise RuntimeError("No usable positive-degradation results found. Run Cell 6 first.")

results_df["log_degradation"] = np.log10(results_df["degradation"])
results_df["ml_worse"] = results_df["degradation"] > 1.0
results_df["scip_limited"] = results_df["scip_status"].astype(str).str.contains("limit", case=False, na=False)
results_df["ml_limited"] = results_df["ml_status"].astype(str).str.contains("limit", case=False, na=False)
classes = sorted(results_df["class"].unique())


def safe_corr(x, y, method="pearson"):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    mask = np.isfinite(x) & np.isfinite(y)
    if mask.sum() < 3 or np.nanstd(x[mask]) == 0 or np.nanstd(y[mask]) == 0:
        return np.nan, np.nan
    if method == "spearman":
        stat = spearmanr(x[mask], y[mask])
        return float(stat.correlation), float(stat.pvalue)
    stat = pearsonr(x[mask], y[mask])
    return float(stat.statistic), float(stat.pvalue)


def binary_auc(scores, labels):
    scores = np.asarray(scores, dtype=float)
    labels = np.asarray(labels, dtype=bool)
    pos = scores[labels]
    neg = scores[~labels]
    if len(pos) == 0 or len(neg) == 0:
        return np.nan
    wins = 0.0
    total = 0
    for p in pos:
        wins += np.sum(p > neg) + 0.5 * np.sum(p == neg)
        total += len(neg)
    return float(wins / total)


def instance_stats(path):
    import pyscipopt

    model = pyscipopt.Model()
    model.hideOutput(True)
    model.readProblem(str(path))
    n_nonzeros = 0
    for cons in model.getConss():
        try:
            n_nonzeros += len(model.getValsLinear(cons))
        except Exception:
            pass
    return {
        "path": str(Path(path).resolve()),
        "n_variables": len(model.getVars()),
        "n_constraints": len(model.getConss()),
        "n_nonzeros": n_nonzeros,
    }

# Table 1: dataset inventory with structural medians.
STATS_CSV = RESULTS_DIR / "instance_stats.csv"
if STATS_CSV.exists():
    stats_df = pd.read_csv(STATS_CSV)
else:
    stats_rows = []
    all_paths = sorted(set(results_df["path"].astype(str)) | {str(p.resolve()) for p in instance_files(REFERENCE_DIR)})
    for idx, path in enumerate(all_paths, start=1):
        if idx % 50 == 0 or idx == 1:
            print(f"instance stats {idx}/{len(all_paths)}")
        try:
            stats_rows.append(instance_stats(path))
        except Exception as exc:
            print(f"stats failed {path}: {type(exc).__name__}: {exc}")
    stats_df = pd.DataFrame(stats_rows)
    stats_df.to_csv(STATS_CSV, index=False)

path_to_class = dict(zip(results_df["path"], results_df["class"]))
for path in instance_files(REFERENCE_DIR):
    path_to_class[str(path.resolve())] = "reference_setcover"
stats_df["class"] = stats_df["path"].map(path_to_class)
dataset_inventory = (
    stats_df.dropna(subset=["class"])
    .groupby("class")
    .agg(
        n_instances=("path", "count"),
        variables_median=("n_variables", "median"),
        constraints_median=("n_constraints", "median"),
        nonzeros_median=("n_nonzeros", "median"),
        variables_min=("n_variables", "min"),
        variables_max=("n_variables", "max"),
    )
    .reset_index()
)
dataset_inventory.to_csv(RESULTS_DIR / "dataset_inventory.csv", index=False)

# Tables 2 and 3: distance and solver summaries.
distance_summary = (
    results_df.groupby("class")["distance"]
    .agg(n="count", median="median", mean="mean", std="std", min="min", max="max")
    .reset_index()
)
distance_summary["iqr"] = results_df.groupby("class")["distance"].quantile(0.75).values - results_df.groupby("class")["distance"].quantile(0.25).values
distance_summary.to_csv(RESULTS_DIR / "distance_summary_by_class.csv", index=False)

solver_summary = (
    results_df.groupby("class")
    .agg(
        n=("path", "count"),
        median_scip_nodes=("scip_nodes", "median"),
        median_ml_nodes=("ml_nodes", "median"),
        median_degradation=("degradation", "median"),
        mean_degradation=("degradation", "mean"),
        pct_ml_worse=("ml_worse", "mean"),
        pct_scip_limited=("scip_limited", "mean"),
        pct_ml_limited=("ml_limited", "mean"),
    )
    .reset_index()
)
if FAILURES_CSV.exists():
    failures_df = pd.read_csv(FAILURES_CSV)
    failure_counts = failures_df.groupby("class").size().rename("failures").reset_index()
    solver_summary = solver_summary.merge(failure_counts, on="class", how="left")
else:
    solver_summary["failures"] = 0
solver_summary["failures"] = solver_summary["failures"].fillna(0).astype(int)
solver_summary.to_csv(RESULTS_DIR / "solver_summary_by_class.csv", index=False)

summary_by_class = distance_summary.merge(solver_summary, on="class", suffixes=("_distance", "_solver"))
summary_by_class.to_csv(RESULTS_DIR / "summary_by_class.csv", index=False)

# Correlation and threshold summaries.
r_raw, p_raw = safe_corr(results_df["distance"], results_df["degradation"], "pearson")
r_log, p_log = safe_corr(results_df["distance"], results_df["log_degradation"], "pearson")
r_spear, p_spear = safe_corr(results_df["distance"], results_df["degradation"], "spearman")
auc = binary_auc(results_df["distance"], results_df["ml_worse"])
correlation_summary = pd.DataFrame([
    {"metric": "pearson_distance_degradation", "r": r_raw, "p_value": p_raw},
    {"metric": "pearson_distance_log10_degradation", "r": r_log, "p_value": p_log},
    {"metric": "spearman_distance_degradation", "r": r_spear, "p_value": p_spear},
    {"metric": "auroc_predict_ml_worse", "r": auc, "p_value": np.nan},
])
correlation_summary.to_csv(RESULTS_DIR / "correlation_summary.csv", index=False)

threshold_rows = []
harmful = results_df["ml_worse"].to_numpy(dtype=bool)
distances = results_df["distance"].to_numpy(dtype=float)
thresholds = np.unique(np.quantile(distances, np.linspace(0, 1, min(101, max(2, len(distances))))))
for threshold in thresholds:
    trust = distances <= threshold
    fallback = ~trust
    false_trust = np.sum(trust & harmful) / max(1, np.sum(trust))
    caught_degradation = np.sum(fallback & harmful) / max(1, np.sum(harmful))
    fallback_rate = np.mean(fallback)
    fallback_precision = np.sum(fallback & harmful) / max(1, np.sum(fallback))
    false_fallback = np.sum(fallback & ~harmful) / max(1, np.sum(~harmful))
    youden_j = caught_degradation - false_fallback
    threshold_rows.append({
        "threshold": float(threshold),
        "trusted_count": int(np.sum(trust)),
        "fallback_count": int(np.sum(fallback)),
        "coverage_trusted": float(np.mean(trust)),
        "fallback_rate": float(fallback_rate),
        "false_trust_rate": float(false_trust),
        "caught_degradation_rate": float(caught_degradation),
        "fallback_precision_pct_degraded": float(fallback_precision),
        "false_fallback_rate": float(false_fallback),
        "youden_j": float(youden_j),
    })
threshold_table = pd.DataFrame(threshold_rows)
threshold_table.to_csv(RESULTS_DIR / "threshold_table.csv", index=False)
best_threshold = threshold_table.sort_values(["youden_j", "caught_degradation_rate"], ascending=False).iloc[0]

leave_rows = []
for held_out in classes:
    subset = results_df[results_df["class"] != held_out]
    rr, pp = safe_corr(subset["distance"], subset["log_degradation"], "pearson")
    sr, sp = safe_corr(subset["distance"], subset["degradation"], "spearman")
    leave_rows.append({"held_out_class": held_out, "pearson_log_r": rr, "pearson_log_p": pp, "spearman_r": sr, "spearman_p": sp, "n": len(subset)})
leave_one_class_out = pd.DataFrame(leave_rows)
leave_one_class_out.to_csv(RESULTS_DIR / "leave_one_class_out.csv", index=False)

within_rows = []
for class_name in classes:
    subset = results_df[results_df["class"] == class_name]
    rr, pp = safe_corr(subset["distance"], subset["log_degradation"], "pearson")
    sr, sp = safe_corr(subset["distance"], subset["degradation"], "spearman")
    within_rows.append({"class": class_name, "pearson_log_r": rr, "pearson_log_p": pp, "spearman_r": sr, "spearman_p": sp, "n": len(subset), "unique_distances": subset["distance"].nunique()})
within_class = pd.DataFrame(within_rows)
within_class.to_csv(RESULTS_DIR / "within_class_correlations.csv", index=False)

print("Correlation summary")
print(correlation_summary.to_string(index=False))
print("\nBest threshold")
print(best_threshold.to_frame().T.to_string(index=False))
print("\nSummary by class")
print(summary_by_class.to_string(index=False))

# Plot helpers.
def savefig(name: str):
    path = RESULTS_DIR / name
    plt.tight_layout()
    plt.savefig(path, dpi=200)
    plt.show()
    print(f"saved {path}")

# Plot 1: headline scatter, log y-axis.
plt.figure(figsize=(10, 7))
for class_name in classes:
    subset = results_df[results_df["class"] == class_name]
    plt.scatter(subset["distance"], subset["degradation"], label=class_name, alpha=0.75)
x = results_df["distance"].to_numpy(dtype=float)
y_log = results_df["log_degradation"].to_numpy(dtype=float)
if np.nanstd(x) > 0 and np.nanstd(y_log) > 0:
    coef = np.polyfit(x, y_log, deg=1)
    x_line = np.linspace(np.nanmin(x), np.nanmax(x), 200)
    y_line = 10 ** (coef[0] * x_line + coef[1])
    plt.plot(x_line, y_line, color="black", linewidth=2, label="linear fit on log10 ratio")
plt.axhline(1.0, color="red", linestyle="--", linewidth=1.5)
plt.yscale("log")
plt.xlabel("Mean Maudet distance to setcover reference")
plt.ylabel("Degradation ratio: ML nodes / vanilla SCIP nodes")
plt.title("MILP Distance vs ML Branching Degradation")
plt.text(0.02, 0.98, f"Pearson log r={r_log:.3f}\nSpearman rho={r_spear:.3f}\nAUROC={auc:.3f}", transform=plt.gca().transAxes, va="top")
plt.legend(fontsize=8)
savefig("scatter_distance_degradation_log.png")

# Plot 2: linear scatter for raw outliers.
plt.figure(figsize=(10, 7))
for class_name in classes:
    subset = results_df[results_df["class"] == class_name]
    plt.scatter(subset["distance"], subset["degradation"], label=class_name, alpha=0.75)
plt.axhline(1.0, color="red", linestyle="--", linewidth=1.5)
plt.xlabel("Mean Maudet distance to setcover reference")
plt.ylabel("Degradation ratio: ML nodes / vanilla SCIP nodes")
plt.title("MILP Distance vs ML Branching Degradation (linear scale)")
plt.legend(fontsize=8)
savefig("scatter_distance_degradation_linear.png")

# Plot 3: distance distribution by class.
order = results_df.groupby("class")["distance"].median().sort_values().index.tolist()
plt.figure(figsize=(11, 6))
plt.boxplot([results_df.loc[results_df["class"] == cls, "distance"] for cls in order], labels=order, showfliers=True)
plt.xticks(rotation=35, ha="right")
plt.ylabel("Mean Maudet distance")
plt.title("Distance to Setcover Reference by Class")
savefig("box_distance_by_class.png")

# Plot 4: degradation distribution by class.
plt.figure(figsize=(11, 6))
plt.boxplot([results_df.loc[results_df["class"] == cls, "degradation"] for cls in order], labels=order, showfliers=True)
plt.axhline(1.0, color="red", linestyle="--", linewidth=1.5)
plt.yscale("log")
plt.xticks(rotation=35, ha="right")
plt.ylabel("Degradation ratio: ML nodes / vanilla SCIP nodes")
plt.title("ML Branching Degradation by Class")
savefig("box_degradation_by_class.png")

# Plot 5: class-median distance vs degradation.
class_medians = results_df.groupby("class").agg(distance=("distance", "median"), degradation=("degradation", "median")).reset_index()
plt.figure(figsize=(8, 6))
plt.scatter(class_medians["distance"], class_medians["degradation"], s=80)
for _, row in class_medians.iterrows():
    plt.annotate(row["class"], (row["distance"], row["degradation"]), xytext=(5, 4), textcoords="offset points", fontsize=8)
plt.axhline(1.0, color="red", linestyle="--", linewidth=1.5)
plt.yscale("log")
plt.xlabel("Class median distance")
plt.ylabel("Class median degradation")
plt.title("Class-Level Distance vs Degradation")
savefig("class_median_distance_vs_degradation.png")

# Plot 6: gate curve.
plt.figure(figsize=(9, 6))
plt.plot(threshold_table["threshold"], threshold_table["false_trust_rate"], label="false trust rate")
plt.plot(threshold_table["threshold"], threshold_table["caught_degradation_rate"], label="caught degradation rate")
plt.plot(threshold_table["threshold"], threshold_table["fallback_rate"], label="fallback rate")
plt.axvline(best_threshold["threshold"], color="black", linestyle=":", label="best threshold")
plt.xlabel("Distance threshold")
plt.ylabel("Rate")
plt.title("Distance Gate Tradeoffs")
plt.legend()
savefig("gate_curve.png")

# Plot 7: confusion matrix at best threshold.
threshold = float(best_threshold["threshold"])
trust = results_df["distance"].to_numpy(dtype=float) <= threshold
fallback = ~trust
harm = results_df["ml_worse"].to_numpy(dtype=bool)
cm = np.array([
    [np.sum(trust & ~harm), np.sum(trust & harm)],
    [np.sum(fallback & ~harm), np.sum(fallback & harm)],
])
plt.figure(figsize=(6, 5))
plt.imshow(cm, cmap="Blues")
plt.xticks([0, 1], ["ML ok", "ML worse"])
plt.yticks([0, 1], ["trust ML", "fallback SCIP"])
for i in range(2):
    for j in range(2):
        plt.text(j, i, int(cm[i, j]), ha="center", va="center", fontsize=14)
plt.title(f"Gate Confusion Matrix at threshold={threshold:.4f}")
plt.colorbar(label="count")
savefig("gate_confusion_matrix.png")

# Plot 8: SCIP nodes vs ML nodes.
plt.figure(figsize=(8, 7))
for class_name in classes:
    subset = results_df[results_df["class"] == class_name]
    plt.scatter(subset["scip_nodes"].clip(lower=1), subset["ml_nodes"].clip(lower=1), label=class_name, alpha=0.75)
lims = [1, max(results_df["scip_nodes"].max(), results_df["ml_nodes"].max(), 2)]
plt.plot(lims, lims, color="red", linestyle="--", linewidth=1.5)
plt.xscale("log")
plt.yscale("log")
plt.xlabel("Vanilla SCIP nodes")
plt.ylabel("ML branching nodes")
plt.title("Vanilla SCIP vs ML Branching Node Counts")
plt.legend(fontsize=8)
savefig("scip_vs_ml_nodes.png")

# Plot 9: distance vs raw node counts.
fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharex=True)
for class_name in classes:
    subset = results_df[results_df["class"] == class_name]
    axes[0].scatter(subset["distance"], subset["scip_nodes"].clip(lower=1), label=class_name, alpha=0.75)
    axes[1].scatter(subset["distance"], subset["ml_nodes"].clip(lower=1), label=class_name, alpha=0.75)
axes[0].set_title("Distance vs vanilla SCIP nodes")
axes[1].set_title("Distance vs ML branching nodes")
for ax in axes:
    ax.set_xlabel("Mean Maudet distance")
    ax.set_yscale("log")
axes[0].set_ylabel("Node count")
axes[1].legend(fontsize=8)
plt.tight_layout()
path = RESULTS_DIR / "distance_vs_nodes.png"
plt.savefig(path, dpi=200)
plt.show()
print(f"saved {path}")

# Plot 10: within-class small multiples.
ncols = 2
nrows = math.ceil(len(classes) / ncols)
fig, axes = plt.subplots(nrows, ncols, figsize=(12, 3.5 * nrows), squeeze=False)
for ax, class_name in zip(axes.ravel(), classes):
    subset = results_df[results_df["class"] == class_name]
    ax.scatter(subset["distance"], subset["degradation"], alpha=0.75)
    ax.axhline(1.0, color="red", linestyle="--", linewidth=1)
    ax.set_yscale("log")
    ax.set_title(class_name)
    ax.set_xlabel("distance")
    ax.set_ylabel("degradation")
for ax in axes.ravel()[len(classes):]:
    ax.axis("off")
plt.tight_layout()
path = RESULTS_DIR / "within_class_panels.png"
plt.savefig(path, dpi=200)
plt.show()
print(f"saved {path}")

# Plot 11: distance histogram.
plt.figure(figsize=(9, 5))
plt.hist(results_df["distance"], bins=30, edgecolor="black")
plt.xlabel("Mean Maudet distance")
plt.ylabel("Count")
plt.title("Distance Histogram")
savefig("distance_histogram.png")
